<a href="https://colab.research.google.com/github/NeuronEdge67/Hands-On-Machine-Learning-...By-Aur-lien-G-ron-/blob/main/12_deep_computer_vision_with_cnns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Deep Computer Vision Using Convolutional Neural Networks**

In [20]:
import numpy as np
import torch
from sklearn.datasets import load_sample_images
import matplotlib.pyplot as plt

In [21]:
sample_images = np.stack(load_sample_images()["images"])
sample_images = torch.tensor(sample_images, dtype=torch.float32) / 255

In [22]:
sample_images.shape

torch.Size([2, 427, 640, 3])

In [23]:
sample_images_permuted = sample_images.permute(0, 3, 1, 2)
sample_images_permuted.shape

torch.Size([2, 3, 427, 640])

In [24]:
import torchvision

import torchvision.transforms.v2 as T

cropped_images = T.CenterCrop((70, 120))(sample_images_permuted)

cropped_images.shape

torch.Size([2, 3, 70, 120])

In [25]:
import torch.nn as nn

torch.manual_seed(42)
conv_layer = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7, padding="same")
fmaps = conv_layer(cropped_images)

In [26]:
fmaps.shape

torch.Size([2, 32, 70, 120])

In [27]:
print(conv_layer.weight.shape)
print(conv_layer.bias.shape)

torch.Size([32, 3, 7, 7])
torch.Size([32])


In [35]:
max_pool = nn.MaxPool2d(kernel_size=2)

In [30]:
# depth wise pooling

import  torch.nn.functional as F

class DepthMaxPool2(torch.nn.Module):
  def __init__(self, kernel_size, stride=None, padding=0):
    super().__init__()
    self.kernel_size = kernel_size
    self.stride = stride
    self.padding = padding

  def forward(self, inputs):
    batch, channels, height, width = inputs.shape
    Z = inputs.view(batch, channels, height * width)
    Z = Z.permute(0, 2, 1)
    Z = F.max_pool1d(Z,  kernel_size=self.kernel_size, stride=self.stride,
                     padding=self.padding)
    Z = Z.permute(0, 2, 1)
    return Z.view(batch, -1, height * width)

In [31]:
global_avg_pool = nn.AvgPool2d(kernel_size=(70, 120))

In [32]:
output = global_avg_pool(cropped_images)

In [33]:
global_avg_pool = nn.AdaptiveAvgPool2d(output_size=1)

In [34]:
output = global_avg_pool(cropped_images)